# COMP8240 — feasibility check for the Min-K% Prob replication

Runs the original code from Shi et al. (ICLR 2024) on the original WikiMIA data,
end to end, so the proposal can honestly say the software was downloaded and executed.

**Before you run anything: `Runtime → Change runtime type → T4 GPU`.**
It works on CPU too, but 776 passages × three model passes on two vCPUs takes
20+ minutes, versus 1–2 minutes on a T4.

Then `Runtime → Run all`. Every long cell streams a live progress bar, so you can
always tell whether it is moving.

The results cell prints the four numbers to paste into Section 4.5 of the proposal:
model, runtime, Min-20% Prob AUC, and PPL baseline AUC.

## 1. Clone the original repository

In [ ]:
%cd -q /content
!git clone --quiet https://github.com/swj0419/detect-pretrain-code.git 2>/dev/null || echo 'already cloned'
!cd /content/detect-pretrain-code && git log -1 --format='commit %h  (%ad)' --date=short
!ls /content/detect-pretrain-code/src

## 2. Pin the environment

The repository ships no `requirements.txt`, so we pin versions ourselves — this is
issue (i) noted in Section 4.2 of the proposal. `accelerate` is needed because
`run.py` loads models with `device_map='auto'`; `openai` is needed only because
`run.py` imports it unconditionally, even on the local-model path.

If pip spends minutes resolving or asks you to restart, drop the `==` pins and
re-run — the script is not fussy about exact versions.

In [ ]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" "accelerate>=0.33" openai scikit-learn matplotlib tqdm
import transformers, datasets, sklearn, torch
print('torch       ', torch.__version__, '| GPU available:', torch.cuda.is_available())
print('transformers', transformers.__version__)
print('datasets    ', datasets.__version__)
print('sklearn     ', sklearn.__version__)
if not torch.cuda.is_available():
    print('\n*** No GPU. This will still work but takes 20+ minutes.')
    print('*** Runtime -> Change runtime type -> T4 GPU, then Run all again.')

## 3. Check the original data downloads

WikiMIA is 469 KB in total. The four splits should report 776 / 542 / 250 / 82 examples.

In [ ]:
from datasets import load_dataset
from collections import Counter

for L in [32, 64, 128, 256]:
    d = load_dataset('swj0419/WikiMIA', split=f'WikiMIA_length{L}')
    print(f'length {L:>3}: n={len(d):>4}  labels={dict(Counter(d["label"]))}  columns={d.column_names}')

d32 = load_dataset('swj0419/WikiMIA', split='WikiMIA_length32')
print('\nexample non-member (label 0):\n ', d32[0]['input'][:200])

## 4. Run the original pipeline

`--target_model` is the model under attack; `--ref_model` is only used by the
*Smaller Ref* baseline. Output streams live — you should see a tqdm bar over 776
examples within a minute of the model finishing downloading.

Once this works, swap in `EleutherAI/pythia-1.4b` or `EleutherAI/pythia-2.8b`.

In [ ]:
TARGET = 'EleutherAI/pythia-160m'
REF    = 'EleutherAI/pythia-70m'
LENGTH = 32

import time, os
os.chdir('/content/detect-pretrain-code/src')
t0 = time.time()
!python run.py --target_model {TARGET} --ref_model {REF} --data swj0419/WikiMIA --length {LENGTH}
RUNTIME = time.time() - t0
print(f'\nwall clock: {RUNTIME/60:.1f} min')

## 5. Read the results the code wrote

`eval.py` writes `auc.txt` and the ROC figure `auc.png` into its output directory.

In [ ]:
import glob, re, os

# NOTE: the repo ships a committed out/ directory holding the AUTHORS' own
# LLaMA-13B results. Match on our own model names so we never read theirs.
run_dir = f'/content/detect-pretrain-code/src/out/{TARGET}_{REF}/input'
auc_path = os.path.join(run_dir, 'auc.txt')
assert os.path.exists(auc_path), (
    f'no auc.txt at {auc_path} — the run in cell 4 did not finish.\n'
    f'other auc.txt files present (these are NOT yours): '
    f'{glob.glob("/content/detect-pretrain-code/src/out/**/auc.txt", recursive=True)}')
OUT_DIR = run_dir
print('output dir:', OUT_DIR, '\n')
text = open(auc_path).read()
print(text)

def auc_for(prefix):
    for line in text.strip().splitlines():
        if line.startswith(prefix):
            return float(re.search(r'AUC ([0-9.]+)', line).group(1))
    return None

mink20 = auc_for('Min_20.0% Prob')
ppl    = auc_for('ppl')

print('=' * 62)
print('PASTE THESE INTO SECTION 4.5 OF THE PROPOSAL')
print('=' * 62)
print(f'  model        : {TARGET}')
print(f'  runtime      : {RUNTIME/60:.1f} minutes')
print(f'  Min-20% AUC  : {mink20:.3f}')
print(f'  PPL baseline : {ppl:.3f}')


In [ ]:
from IPython.display import Image, display
display(Image(filename=os.path.join(OUT_DIR, 'auc.png')))

## 6. Independent re-implementation (sanity check)

Thirty lines that recompute Min-K% Prob from scratch, with its own progress bar.
If this agrees with the repository's number, the sign convention noted as issue (iii)
in Section 4.2 has been read correctly; if it comes out at `1 - AUC`, the ROC curve
is inverted. This cell also works standalone if cell 4 ever fails.

In [ ]:
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(TARGET)
model = AutoModelForCausalLM.from_pretrained(TARGET).to(device).eval()

def token_logprobs(text):
    ids = torch.tensor(tok.encode(text)).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(ids).logits
    lp = torch.log_softmax(logits, dim=-1)[0]
    return [lp[i, t].item() for i, t in enumerate(ids[0][1:])]

def min_k_prob(text, k=0.2):
    lps = token_logprobs(text)
    n = max(1, int(len(lps) * k))
    return float(np.mean(sorted(lps)[:n]))   # higher (less negative) => member

scores = [min_k_prob(ex['input']) for ex in tqdm(d32, desc='scoring')]
labels = [ex['label'] for ex in d32]
print(f'independent Min-20% AUC = {roc_auc_score(labels, scores):.3f}')
try:
    print(f'repository  Min-20% AUC = {mink20:.3f}')
except NameError:
    print('(repository run not available — use the number above)')